Copyright 2025 Google LLC<br>
<br>
Licensed under the Apache License, Version 2.0 (the "License");<br>
you may not use this file except in compliance with the License.<br>
You may obtain a copy of the License at<br>
<br>
    http://www.apache.org/licenses/LICENSE-2.0<br>
<br>
Unless required by applicable law or agreed to in writing, software<br>
distributed under the License is distributed on an "AS IS" BASIS,<br>
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.<br>
See the License for the specific language governing permissions and<br>
limitations under the License.


Critic agent for identifying and verifying statements using search tools.
<br>
import os<br>
from google.adk import Agent<br>
from google.adk.agents.callback_context import CallbackContext<br>
from google.adk.models import LlmResponse<br>
from google.adk.tools import google_search<br>
from google.genai import types<br>
import sys<br>
sys.path.append("../..")<br>
from callback_logging import log_query_to_model, log_model_response<br>
from . import prompt<br>
def _render_reference(<br>
    callback_context: CallbackContext,<br>
    llm_response: LlmResponse,<br>
) -> LlmResponse:<br>
  
Appends grounding references to the response.


In [ ]:
    del callback_context
    if (
        not llm_response.content or
        not llm_response.content.parts or
        not llm_response.grounding_metadata
    ):
        return llm_response
    references = []
    for chunk in llm_response.grounding_metadata.grounding_chunks or []:
        title, uri, text = '', '', ''
        if chunk.retrieved_context:
            title = chunk.retrieved_context.title
            uri = chunk.retrieved_context.uri
            text = chunk.retrieved_context.text
        elif chunk.web:
            title = chunk.web.title
            uri = chunk.web.uri
        parts = [s for s in (title, text) if s]
        if uri and parts:
            parts[0] = f'[{parts[0]}]({uri})'
        if parts:
            references.append('* ' + ': '.join(parts) + '\n')
    if references:
        reference_text = ''.join(['\n\nReference:\n\n'] + references)
        llm_response.content.parts.append(types.Part(text=reference_text))
    if all(part.text is not None for part in llm_response.content.parts):
        all_text = '\n'.join(part.text for part in llm_response.content.parts)
        llm_response.content.parts[0].text = all_text
        del llm_response.content.parts[1:]
    return llm_response

In [ ]:
critic_agent = Agent(
    model=os.getenv("MODEL"),
    name='critic_agent',
    instruction=prompt.CRITIC_PROMPT,
    tools=[google_search],
    before_model_callback=log_query_to_model,
    after_model_callback=_render_reference,

In [ ]:
)